# Model Selection

In [18]:
import importlib
from Config import constants
import pytorch_lightning as pl
import pandas as pd
from src.preprocessing.GTSRBDataset import GTSRBDataset
from src.preprocessing.transformations import get_transforms
from torch.utils.data import DataLoader
from src.models.model_template import TrafficSignClassifier
from src.models.experiment_manager import get_experiment_setup
import yaml
from pathlib import Path
from sklearn.model_selection import GroupKFold
import numpy as np
from pytorch_lightning.callbacks import EarlyStopping

importlib.reload(constants)

<module 'Config.constants' from 'F:\\Software\\Dataspell\\TrafficSigns\\Config\\constants.py'>

## Pick top 3 models by accuracy for further cross validation

In [19]:
test_df = pd.read_csv(constants.TEST_CSV_PATH)

test_ds = GTSRBDataset(test_df, transform=get_transforms(224, train=False))

test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

In [20]:
trainer = pl.Trainer(
    max_epochs=100,
    accelerator="auto",
    check_val_every_n_epoch=1,
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [21]:
log_dir = Path(constants.LOGS_DIR)

configs = []

for hparams_path in log_dir.rglob("hparams.yaml"):
    with open(hparams_path, "r") as f:
        config = yaml.safe_load(f)

    # Optional: add experiment name
    config["experiment_name"] = hparams_path.parent.parent.name
    config["version"] = hparams_path.parent.name

    configs.append(config)

In [22]:
models = []
results_list = []

for ckpt_path, config in zip(constants.MODEL_CHECKPOINTS_DIR.glob("*.ckpt"), configs):
    exp_setup = get_experiment_setup(config, constants.NUM_CLASSES)

    optimizer = exp_setup["optimizer"]
    criterion = exp_setup["criterion"]
    model_template = exp_setup["model"]

    model = TrafficSignClassifier.load_from_checkpoint(
        ckpt_path,
        num_classes=constants.NUM_CLASSES,
        model=model_template,
        optimizer=optimizer,
        criterion=criterion,
    )

    trainer.test(model, dataloaders=test_loader)

    test_result = trainer.test(model, dataloaders=test_loader, verbose=False)[0]

    results_list.append({
        "config": model.config,   # or just config
        "ckpt_path": ckpt_path,
        "test_acc": test_result["test_acc"],
        "test_loss": test_result["test_loss"]
    })

F:\Software\Dataspell\TrafficSigns\venv\lib\site-packages\pytorch_lightning\utilities\parsing.py:208: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.
F:\Software\Dataspell\TrafficSigns\venv\lib\site-packages\pytorch_lightning\utilities\parsing.py:208: Attribute 'criterion' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['criterion'])`.
You are using a CUDA device ('NVIDIA GeForce RTX 3070') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
F:\Software\Dataspell\Traffi

Testing: |          | 0/? [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc            0.6914728879928589
        test_loss           1.2846468687057495
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


Testing: |          | 0/? [00:00<?, ?it/s]

...

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc            0.9689922332763672
        test_loss           0.14094385504722595
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


Testing: |          | 0/? [00:00<?, ?it/s]

In [23]:
top3 = sorted(results_list, key=lambda x: x["test_acc"], reverse=True)[:3]

In [24]:
for i, res in enumerate(top3):
    print(f"Acc: {res['test_acc']:.4f}")
    print(f"Checkpoint: {res['ckpt_path']}")
    print("-" * 30)

Acc: 0.9798
Checkpoint: F:\Software\Dataspell\TrafficSigns\Experiments\checkpoints\exp005-epoch=36-val_acc=0.995.ckpt
------------------------------
Acc: 0.9791
Checkpoint: F:\Software\Dataspell\TrafficSigns\Experiments\checkpoints\exp004-epoch=18-val_acc=0.995.ckpt
------------------------------
Acc: 0.9690
Checkpoint: F:\Software\Dataspell\TrafficSigns\Experiments\checkpoints\exp010-epoch=99-val_acc=0.986.ckpt
------------------------------


## Cross Validation

In [26]:
df = pd.read_csv(constants.ALL_IMAGES_CSV_PATH)

In [27]:
df["track_id"] = df.groupby("ClassId").cumcount() // 30

In [28]:
test_tracks = []

for class_id, group in df.groupby("ClassId"):
    unique_tracks = group["track_id"].unique()
    test_track = unique_tracks[0]  # or random.choice
    test_tracks.append((class_id, test_track))

test_df = pd.concat([
    df[(df["ClassId"] == cid) & (df["track_id"] == tid)]
    for cid, tid in test_tracks
])

In [29]:
trainval_df = df.drop(test_df.index)

In [30]:
gkf = GroupKFold(n_splits=4)

splits = list(gkf.split(
    trainval_df,
    y=trainval_df["ClassId"],
    groups=trainval_df["track_id"]
))

In [31]:
out_dir = Path(constants.FOLDS_DIR)
out_dir.mkdir(exist_ok=True)

for fold, (train_idx, val_idx) in enumerate(splits):
    train_df = trainval_df.iloc[train_idx]
    val_df   = trainval_df.iloc[val_idx]

    train_df.to_csv(out_dir / f"fold_{fold}_train.csv", index=False)
    val_df.to_csv(out_dir / f"fold_{fold}_val.csv", index=False)

test_df.to_csv(out_dir / "test.csv", index=False)

In [35]:
all_results = []

early_stop_callback = EarlyStopping(
    monitor="val_acc",
    mode="max",
    patience=15,
    verbose=True,
)

for config in top3:
    cfg = config["config"]

    fold_results = []

    IMAGE_SIZE = cfg["image_size"]
    BATCH_SIZE = cfg["batch_size"]

    test_ds  = GTSRBDataset(test_df, transform=get_transforms(IMAGE_SIZE, train=False))
    test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    for fold, (train_idx, val_idx) in enumerate(splits):

        train_df = trainval_df.iloc[train_idx]
        val_df   = trainval_df.iloc[val_idx]

        # datasets
        train_ds = GTSRBDataset(train_df, transform=get_transforms(IMAGE_SIZE))
        val_ds   = GTSRBDataset(val_df, transform=get_transforms(IMAGE_SIZE, train=False))

        # loaders
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

        # model
        experiment_setup = get_experiment_setup(cfg, num_classes=constants.NUM_CLASSES)

        model = experiment_setup["model"]
        optimizer = experiment_setup["optimizer"]
        criterion = experiment_setup["criterion"]

        model = TrafficSignClassifier(cfg, model, criterion, optimizer)

        trainer = pl.Trainer(
            max_epochs=100,
            callbacks=[early_stop_callback],
            accelerator="auto",
            check_val_every_n_epoch=1,
        )

        # train
        trainer.fit(model, train_loader, val_loader)

        # validate
        val_metrics = trainer.validate(model, val_loader, verbose=False)[0]

        # test (optional per fold)
        test_metrics = trainer.test(model, test_loader, verbose=False)[0]

        fold_results.append({
            "fold": fold,
            "val_acc": val_metrics["val_acc"],
            "test_acc": test_metrics["test_acc"]
        })

    # aggregate per config
    all_results.append({
        "config": config,
        "mean_val_acc": np.mean([r["val_acc"] for r in fold_results]),
        "mean_test_acc": np.mean([r["test_acc"] for r in fold_results]),
        "std_val_acc": np.std([r["val_acc"] for r in fold_results]),
    })

F:\Software\Dataspell\TrafficSigns\venv\lib\site-packages\pytorch_lightning\utilities\parsing.py:208: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.
F:\Software\Dataspell\TrafficSigns\venv\lib\site-packages\pytorch_lightning\utilities\parsing.py:208: Attribute 'criterion' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['criterion'])`.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type             | Params | Mode 
-------------------------------------------------------
0 | model     | EfficientNet     | 4.1 M  | train
1 | criterion | CrossEntropyLoss | 0      | train
-------------------------------------------------------
4.1 M     Tra

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

F:\Software\Dataspell\TrafficSigns\venv\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
F:\Software\Dataspell\TrafficSigns\venv\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:424: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_acc did not improve in the last 26 records. Best score: 0.989. Signaling Trainer to stop.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

## Model Selection

In [40]:
for result in all_results:
    print(result["mean_val_acc"], result["mean_test_acc"], result["std_val_acc"])
    print("-"*50)

0.9073296040296555 0.8443798422813416 0.040318159621571696
--------------------------------------------------
0.8912768363952637 0.8410852700471878 0.049567427804654944
--------------------------------------------------
0.47380727529525757 0.3408914655447006 0.05187264951875422
--------------------------------------------------


In [45]:
config_data = all_results[0]["config"]["config"]

to_exclude = {"model", "optimizer", "criterion"}

filtered_params = {k: v for k, v in config_data.items() if k not in to_exclude}

filtered_params

{'model_name': 'efficientnet_b0',
 'image_size': 224,
 'batch_size': 48,
 'lr': 0.0003,
 'freeze_backbone': True,
 'unfreeze_last_n': 6,
 'optimizer_name': 'adam',
 'loss_name': 'cross_entropy',
 'dropout': 0.3,
 'weight_decay': 0.0001,
 'num_classes': 43}

## Final model will be config from experiment 5